In [1]:
# For tips on running notebooks in Google Colab, see
# https://docs.pytorch.org/tutorials/beginner/colab
%matplotlib inline

[Learn the Basics](intro.html) \|\|
[Quickstart](quickstart_tutorial.html) \|\|
[Tensors](tensorqs_tutorial.html) \|\| [Datasets &
DataLoaders](data_tutorial.html) \|\| **Transforms** \|\| [Build
Model](buildmodel_tutorial.html) \|\|
[Autograd](autogradqs_tutorial.html) \|\|
[Optimization](optimization_tutorial.html) \|\| [Save & Load
Model](saveloadrun_tutorial.html)

Transforms
==========

Data does not always come in its final processed form that is required
for training machine learning algorithms. We use **transforms** to
perform some manipulation of the data and make it suitable for training.

All TorchVision datasets have two parameters -`transform` to modify the
features and `target_transform` to modify the labels - that accept
callables containing the transformation logic. The
[torchvision.transforms](https://pytorch.org/vision/stable/transforms.html)
module offers several commonly-used transforms out of the box.

The FashionMNIST features are in PIL Image format, and the labels are
integers. For training, we need the features as normalized tensors, and
the labels as one-hot encoded tensors. To make these transformations, we
use `ToTensor` and `Lambda`.


In [2]:
import torch
from torchvision import datasets
from torchvision.transforms import ToTensor, Lambda

ds = datasets.FashionMNIST(
    root="data",
    train=True,
    download=True,
    transform=ToTensor(),
    target_transform=Lambda(lambda y: torch.zeros(10, dtype=torch.float).scatter_(0, torch.tensor(y), value=1))
)

100%|██████████| 26.4M/26.4M [00:01<00:00, 18.6MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 501kB/s]
100%|██████████| 4.42M/4.42M [00:00<00:00, 6.27MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 19.4MB/s]


ToTensor()
==========

[ToTensor](https://pytorch.org/vision/stable/transforms.html#torchvision.transforms.ToTensor)
converts a PIL image or NumPy `ndarray` into a `FloatTensor`. and scales
the image\'s pixel intensity values in the range \[0., 1.\]


Lambda Transforms
=================

Lambda transforms apply any user-defined lambda function. Here, we
define a function to turn the integer into a one-hot encoded tensor. It
first creates a zero tensor of size 10 (the number of labels in our
dataset) and calls
[scatter\_](https://pytorch.org/docs/stable/generated/torch.Tensor.scatter_.html)
which assigns a `value=1` on the index as given by the label `y`.


In [3]:
target_transform = Lambda(lambda y: torch.zeros(
    10, dtype=torch.float).scatter_(dim=0, index=torch.tensor(y), value=1))

In [4]:
print("=== EXPERIMENT 1: Manual One-Hot Encoding ===")
def manual_one_hot(label, num_classes=10):
    """Create one-hot encoding manually to understand the process"""
    one_hot = torch.zeros(num_classes)
    one_hot[label] = 1
    return one_hot

# Test with different labels
labels = [0, 3, 7, 9]
for label in labels:
    one_hot = manual_one_hot(label)
    print(f"Label {label} → {one_hot}")

=== EXPERIMENT 1: Manual One-Hot Encoding ===
Label 0 → tensor([1., 0., 0., 0., 0., 0., 0., 0., 0., 0.])
Label 3 → tensor([0., 0., 0., 1., 0., 0., 0., 0., 0., 0.])
Label 7 → tensor([0., 0., 0., 0., 0., 0., 0., 1., 0., 0.])
Label 9 → tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 1.])


In [5]:
print("\n=== EXPERIMENT 2: Understanding scatter_ ===")
def demonstrate_scatter(label):
    """Show step-by-step what scatter_ does"""
    print(f"\nConverting label {label}:")

    # Step 1: Create zeros tensor
    zeros = torch.zeros(10, dtype=torch.float)
    print(f"Step 1 - zeros tensor: {zeros}")

    # Step 2: Convert label to tensor
    label_tensor = torch.tensor(label)
    print(f"Step 2 - label as tensor: {label_tensor}")

    # Step 3: Use scatter_
    result = zeros.scatter_(0, label_tensor, value=1)
    print(f"Step 3 - after scatter_: {result}")

    return result

# Test scatter_ with different labels
for label in [1, 5, 8]:
    demonstrate_scatter(label)


=== EXPERIMENT 2: Understanding scatter_ ===

Converting label 1:
Step 1 - zeros tensor: tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0.])
Step 2 - label as tensor: 1
Step 3 - after scatter_: tensor([0., 1., 0., 0., 0., 0., 0., 0., 0., 0.])

Converting label 5:
Step 1 - zeros tensor: tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0.])
Step 2 - label as tensor: 5
Step 3 - after scatter_: tensor([0., 0., 0., 0., 0., 1., 0., 0., 0., 0.])

Converting label 8:
Step 1 - zeros tensor: tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0.])
Step 2 - label as tensor: 8
Step 3 - after scatter_: tensor([0., 0., 0., 0., 0., 0., 0., 0., 1., 0.])


In [6]:
print("\n=== EXPERIMENT 3: Compare Regular vs One-Hot Dataset ===")

# Regular dataset (integer labels)
ds_regular = datasets.FashionMNIST(
    root="data",
    train=True,
    download=True,
    transform=ToTensor()
)

# One-hot encoded dataset
ds_onehot = datasets.FashionMNIST(
    root="data",
    train=True,
    download=True,
    transform=ToTensor(),
    target_transform=Lambda(lambda y: torch.zeros(10, dtype=torch.float).scatter_(0, torch.tensor(y), value=1))
)

print("Comparing first 5 samples:")
for i in range(5):
    img_reg, label_reg = ds_regular[i]
    img_onehot, label_onehot = ds_onehot[i]

    print(f"Sample {i}:")
    print(f"  Regular label: {label_reg} (type: {type(label_reg)})")
    print(f"  One-hot label: {label_onehot} (type: {type(label_onehot)})")
    print(f"  One-hot shape: {label_onehot.shape}")



=== EXPERIMENT 3: Compare Regular vs One-Hot Dataset ===
Comparing first 5 samples:
Sample 0:
  Regular label: 9 (type: <class 'int'>)
  One-hot label: tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 1.]) (type: <class 'torch.Tensor'>)
  One-hot shape: torch.Size([10])
Sample 1:
  Regular label: 0 (type: <class 'int'>)
  One-hot label: tensor([1., 0., 0., 0., 0., 0., 0., 0., 0., 0.]) (type: <class 'torch.Tensor'>)
  One-hot shape: torch.Size([10])
Sample 2:
  Regular label: 0 (type: <class 'int'>)
  One-hot label: tensor([1., 0., 0., 0., 0., 0., 0., 0., 0., 0.]) (type: <class 'torch.Tensor'>)
  One-hot shape: torch.Size([10])
Sample 3:
  Regular label: 3 (type: <class 'int'>)
  One-hot label: tensor([0., 0., 0., 1., 0., 0., 0., 0., 0., 0.]) (type: <class 'torch.Tensor'>)
  One-hot shape: torch.Size([10])
Sample 4:
  Regular label: 0 (type: <class 'int'>)
  One-hot label: tensor([1., 0., 0., 0., 0., 0., 0., 0., 0., 0.]) (type: <class 'torch.Tensor'>)
  One-hot shape: torch.Size([10])


In [9]:
print("\n=== EXPERIMENT 4: Alternative One-Hot Methods ===")

def one_hot_functional(label, num_classes=10):
    """Using torch.nn.functional.one_hot"""
    return torch.nn.functional.one_hot(torch.tensor(label), num_classes=num_classes).float()

def one_hot_eye(label, num_classes=10):
    """Using torch.eye (identity matrix)"""
    return torch.eye(num_classes)[label]

# Compare different methods
test_label = 4
method1 = manual_one_hot(test_label)
method2 = one_hot_functional(test_label)
method3 = one_hot_eye(test_label)

print(f"Label {test_label} encoded different ways:")
print(f"Manual method:     {method1}")
print(f"Functional method: {method2}")
print(f"Eye method:        {method3}")
print(f"All equal? {torch.equal(method1, method2) and torch.equal(method2, method3.float())}")



=== EXPERIMENT 4: Alternative One-Hot Methods ===
Label 4 encoded different ways:
Manual method:     tensor([0., 0., 0., 0., 1., 0., 0., 0., 0., 0.])
Functional method: tensor([0., 0., 0., 0., 1., 0., 0., 0., 0., 0.])
Eye method:        tensor([0., 0., 0., 0., 1., 0., 0., 0., 0., 0.])
All equal? True


In [10]:
def one_hot_to_label(one_hot_tensor):
    """Convert one-hot back to integer label"""
    return torch.argmax(one_hot_tensor).item()

# Test conversion back
one_hot_examples = [
    torch.tensor([1., 0., 0., 0., 0., 0., 0., 0., 0., 0.]),  # Should be 0
    torch.tensor([0., 0., 0., 1., 0., 0., 0., 0., 0., 0.]),  # Should be 3
    torch.tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 1.])   # Should be 9
]

for i, one_hot in enumerate(one_hot_examples):
    original_label = one_hot_to_label(one_hot)
    print(f"One-hot {one_hot} → Label {original_label}")

One-hot tensor([1., 0., 0., 0., 0., 0., 0., 0., 0., 0.]) → Label 0
One-hot tensor([0., 0., 0., 1., 0., 0., 0., 0., 0., 0.]) → Label 3
One-hot tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 1.]) → Label 9


In [11]:
print("\n=== EXPERIMENT 6: Understanding Why We Need One-Hot ===")
print("Imagine training a neural network:")
print("- Network output: [0.1, 0.7, 0.2] (probabilities for 3 classes)")
print("- If target is class 1:")
print("  - Integer label: 1 (hard to compare with probabilities)")
print("  - One-hot label: [0, 1, 0] (easy to compare!)")
print("- We can calculate loss by comparing [0.1, 0.7, 0.2] with [0, 1, 0]")


=== EXPERIMENT 6: Understanding Why We Need One-Hot ===
Imagine training a neural network:
- Network output: [0.1, 0.7, 0.2] (probabilities for 3 classes)
- If target is class 1:
  - Integer label: 1 (hard to compare with probabilities)
  - One-hot label: [0, 1, 0] (easy to compare!)
- We can calculate loss by comparing [0.1, 0.7, 0.2] with [0, 1, 0]


In [13]:
from torch import nn

In [16]:
input = torch.randn(32, 1, 5, 5)
# With default parameters
m = nn.Flatten()
output = m(input)
output.size()
# With non-default parameters
m = nn.Flatten(0, 2)
output = m(input)
output.size()

# I think there was just another method like function inside flatten
# hence we could just send input like that
# tried to demonstrate it below

torch.Size([160, 5])

In [22]:
def s(a = 10,b=20):
  sa = a + b
  return sa

um = s
ans = um(10,12)
ans

22

------------------------------------------------------------------------


Further Reading
===============

-   [torchvision.transforms
    API](https://pytorch.org/vision/stable/transforms.html)
